# HighPitchCT — Stage 0 على Kaggle
فعّل **Internet**. هذه المرحلة لا تحتاج GPU؛ استخدم Accelerator=None للتنزيل والفهرسة. سنستخدم T4 x2 لاحقًا لإعادة البناء والتدريب.


In [ ]:
from pathlib import Path
repo = Path('/kaggle/working/HighPitchCT-Pipeline')
if not repo.exists():
    !git clone https://github.com/BrahimSoufghalem/HighPitchCT-Pipeline.git /kaggle/working/HighPitchCT-Pipeline
%cd /kaggle/working/HighPitchCT-Pipeline
!python -m pip install -q -e .


In [ ]:
!hpct env


## عرّف Pilot
استبدل القيم التي تبدأ بـ `PASTE_` فقط. لا تخمّن UID. الصور المنشورة توسم `reconstruction_published` ولا تستخدم Ground Truth للتدريب.


In [ ]:
import csv
from pathlib import Path

series = [
    {
        'case_id': 'pilot_001',
        'series_instance_uid': 'PASTE_PROJECTION_SERIES_UID',
        'role': 'projection_full',
        'description': 'Full-dose projection series',
        'enabled': 'true',
    },
]
placeholders = [r for r in series if r['series_instance_uid'].startswith('PASTE_')]
if placeholders:
    raise ValueError('استبدل PASTE_PROJECTION_SERIES_UID بقيمة حقيقية قبل المتابعة')
manifest = Path('/kaggle/working/series.csv')
with manifest.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=series[0].keys())
    writer.writeheader()
    writer.writerows(series)
print(manifest.read_text())


In [ ]:
!hpct validate-manifest --series-file /kaggle/working/series.csv


In [ ]:
!hpct download --series-file /kaggle/working/series.csv --workers 2 --retries 5


In [ ]:
!hpct index


In [ ]:
import json
from pathlib import Path
summary_path = Path('/kaggle/working/highpitchct-data/manifests/dicom_index_summary.json')
summary = json.loads(summary_path.read_text())
summary


## شرط التوقف
لا تبدأ Stage 1 إذا كانت `read_errors` غير صفرية أو إذا لم نتأكد يدويًا أن السلسلة هي DICOM-CT-PD projection. احفظ نسخة Notebook وملفات manifests ضمن مخرجات Kaggle.
